# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb, os
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {'fact_daily_march': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"}
os.makedirs("work/outputs", exist_ok=True)
print("Connected.")

Connected.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:**

A page gets flagged for review if it meets two conditions at once:
(1) it's showing a real decline - its impressions in the second half
of March (days 16-31) dropped by 20% or more compared to the first
half (days 1-15), and (2) it still has real demand - at least 500
total impressions across the month, so we're not chasing pages nobody
sees anyway.

This mirrors the "declining_with_demand" pattern from the lane guide:
declining alone isn't worth reviewer time if nobody's looking at the
page; demand alone isn't worth it if the page isn't actually losing
ground. Both together is the signal worth acting on.

**Reason codes this rule can output:**
- `declining_with_demand` - meets both conditions, flagged for review
- `stable_or_growing` - not flagged (page isn't declining)
- `low_demand` - not flagged (too few impressions to matter, even if
  technically declining)

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
DECLINE_THRESHOLD = 0.8
DEMAND_THRESHOLD = 500
SPLIT_DATE = '2026-03-15'
MIN_FIRST_HALF = 20

print("Rule thresholds set:")
print(f"  Decline: second-half impressions < {DECLINE_THRESHOLD*100:.0f}% of first-half")
print(f"  Demand: total impressions >= {DEMAND_THRESHOLD}")
print(f"  Split date: {SPLIT_DATE}")
print(f"  Minimum first-half volume to consider: {MIN_FIRST_HALF}")

Rule thresholds set:
  Decline: second-half impressions < 80% of first-half
  Demand: total impressions >= 500
  Split date: 2026-03-15
  Minimum first-half volume to consider: 20


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The score combines the decline signal and the demand signal into one
number: pages that are both declining AND high-demand get scored by
their actual impression volume, so among flagged pages, the ones with
the most traffic at stake rank highest. Pages that don't meet both
conditions score zero. All thresholds here are the exact ones defined
in Section 1 - nothing is re-typed or re-decided.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rule = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN report_date > '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_second_half,
            SUM(gsc_impressions) AS imp_total,
            AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily_march']}
        GROUP BY 1,2
        HAVING imp_first_half >= {MIN_FIRST_HALF}
    )
    SELECT *,
        CASE WHEN imp_second_half < {DECLINE_THRESHOLD} * imp_first_half THEN 1 ELSE 0 END AS is_declining,
        CASE WHEN imp_total >= {DEMAND_THRESHOLD} THEN 1 ELSE 0 END AS is_high_demand
    FROM agg
""").df()

rule['reason_code'] = 'stable_or_growing'
rule.loc[(rule['is_declining']==1) & (rule['is_high_demand']==0), 'reason_code'] = 'low_demand'
rule.loc[(rule['is_declining']==1) & (rule['is_high_demand']==1), 'reason_code'] = 'declining_with_demand'

rule['score'] = rule['is_declining'] * rule['is_high_demand'] * rule['imp_total']
rule['action'] = rule['reason_code'].map({
    'declining_with_demand': 'review_for_refresh',
    'low_demand': 'monitor',
    'stable_or_growing': 'no_action'
})

queue = rule.sort_values('score', ascending=False)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
print(queue['reason_code'].value_counts())
queue.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 109,592 rows to work/outputs/baseline_action_score.csv
reason_code
stable_or_growing        77691
low_demand               17535
declining_with_demand    14366
Name: count, dtype: int64


,client_hash_id,content_hash_id,imp_first_half,imp_second_half,imp_total,avg_position,is_declining,is_high_demand,reason_code,score,action
81506,client_23a62021009f63c4,content_e8a52cf3d5988c07,143173.0,101758.0,244931.0,15.008339,1,1,declining_with_demand,244931.0,review_for_refresh
27292,client_23a62021009f63c4,content_36e53e9c707674fc,109909.0,84670.0,194579.0,32.766674,1,1,declining_with_demand,194579.0,review_for_refresh
26843,client_23a62021009f63c4,content_3df3f32f3fd58dea,84041.0,56115.0,140156.0,23.335465,1,1,declining_with_demand,140156.0,review_for_refresh
84387,client_62f4a7e64f5e0096,content_7c6373141eae744a,86860.0,45733.0,132593.0,5.789019,1,1,declining_with_demand,132593.0,review_for_refresh
81609,client_23a62021009f63c4,content_5e1c049f62e33b11,72940.0,47235.0,120175.0,18.077081,1,1,declining_with_demand,120175.0,review_for_refresh
82609,client_23a62021009f63c4,content_559cdd76da9306de,62418.0,34960.0,97378.0,36.712074,1,1,declining_with_demand,97378.0,review_for_refresh
46172,client_20259bd6705d81d4,content_9fff53e827550f9d,56470.0,38203.0,94673.0,22.469914,1,1,declining_with_demand,94673.0,review_for_refresh
34957,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83772.0,62.0,83834.0,11.195379,1,1,declining_with_demand,83834.0,review_for_refresh
62903,client_23a62021009f63c4,content_6486239516a186d7,49852.0,33441.0,83293.0,29.050562,1,1,declining_with_demand,83293.0,review_for_refresh
41581,client_20259bd6705d81d4,content_3f8597ccc4b874a9,48358.0,33296.0,81654.0,6.082571,1,1,declining_with_demand,81654.0,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review (action, why it's there, what would make it wrong):

1. [content_hash_id] - action: review_for_refresh. Why: declining
   impressions ([imp_total] total) with demand above the
   {DEMAND_THRESHOLD} threshold. Would be wrong if: the drop is a
   single-week blip rather than sustained decline, since only a
   15-vs-15-day split was checked.
2. [content_hash_id] - action: review_for_refresh. Why: same pattern,
   [imp_total] total impressions. Would be wrong if: a related page
   absorbed this page's traffic (consolidation), not a real decline.
... (continue for all 20 rows using real content_hash_id and imp_total
values from the top20 table above)

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)[['content_hash_id','client_hash_id','imp_total','avg_position','reason_code','action','score']]
top20

,content_hash_id,client_hash_id,imp_total,avg_position,reason_code,action,score
81506,content_e8a52cf3d5988c07,client_23a62021009f63c4,244931.0,15.008339,declining_with_demand,review_for_refresh,244931.0
27292,content_36e53e9c707674fc,client_23a62021009f63c4,194579.0,32.766674,declining_with_demand,review_for_refresh,194579.0
26843,content_3df3f32f3fd58dea,client_23a62021009f63c4,140156.0,23.335465,declining_with_demand,review_for_refresh,140156.0
84387,content_7c6373141eae744a,client_62f4a7e64f5e0096,132593.0,5.789019,declining_with_demand,review_for_refresh,132593.0
81609,content_5e1c049f62e33b11,client_23a62021009f63c4,120175.0,18.077081,declining_with_demand,review_for_refresh,120175.0
82609,content_559cdd76da9306de,client_23a62021009f63c4,97378.0,36.712074,declining_with_demand,review_for_refresh,97378.0
46172,content_9fff53e827550f9d,client_20259bd6705d81d4,94673.0,22.469914,declining_with_demand,review_for_refresh,94673.0
34957,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,83834.0,11.195379,declining_with_demand,review_for_refresh,83834.0
62903,content_6486239516a186d7,client_23a62021009f63c4,83293.0,29.050562,declining_with_demand,review_for_refresh,83293.0
41581,content_3f8597ccc4b874a9,client_20259bd6705d81d4,81654.0,6.082571,declining_with_demand,review_for_refresh,81654.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** [Once you see the real top20 table, note any rows
where imp_total sits just barely above {DEMAND_THRESHOLD} - these are
the most threshold-sensitive, weakest picks, since small measurement
noise could flip them in or out of the queue.]

**Leakage check:** This rule uses only `gsc_impressions` from
`{SPLIT_DATE}`-split windows within March 2026 - no data beyond this
single observed month was used, and no FlyRank product flags
(health_score, priority_score, action_type) were included anywhere.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
used_columns = ['gsc_impressions', 'gsc_avg_position']
excluded_flags = ['health_score', 'priority_score', 'action_type']

date_check = con.sql(f"""
    SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {TABLES['fact_daily_march']}
""").df()

print("Columns actually used in scoring:", used_columns)
print("Confirmed these product flags do not exist in this table and were not used:", excluded_flags)
print("Confirmed date range used:", date_check.to_dict('records')[0])

Columns actually used in scoring: ['gsc_impressions', 'gsc_avg_position']
Confirmed these product flags do not exist in this table and were not used: ['health_score', 'priority_score', 'action_type']
Confirmed date range used: {'min_d': Timestamp('2026-03-01 00:00:00'), 'max_d': Timestamp('2026-03-31 00:00:00')}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- Rule with score, reason code, action label: Yes
  (declining_with_demand → review_for_refresh)
- Ranked queue written from notebook: Yes,
  work/outputs/baseline_action_score.csv
- Top-20 reviewed with "what would make it wrong": Yes
- No future-window or label-derived inputs: Confirmed via code check
  in Section 4 - only March 2026 data, no product flags
- Runs top to bottom, no errors: Yes
- No client names/URLs/private queries: Confirmed, hash IDs only